In [ ]:
## Early Detection of Mental Health Issues Using Social Media Analysis

In [ ]:
# =====================================================
# Import Required Libraries
# =====================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_curve,
    auc
)
from sklearn.preprocessing import label_binarize

In [ ]:
# ============================================
#  Load Dataset
# ============================================

df = pd.read_csv('data/mental_health.csv')

# Display first 5 rows
df.head()

In [ ]:
# Data Cleaning

df = df.fillna("")

In [ ]:
# Featue Engineering

df["text_length"] = df["statement"].apply(len)
df["word_count"] = df["statement"].apply(lambda x: len(x.split()))

df[["text_length", "word_count"]].describe()

In [ ]:
## Exploratory Data Analysis

In [ ]:
## Univariate

# Status Distribution
plt.figure(figsize=(6,4))
sns.countplot(x="status", data=df)
plt.title("Mental Health Status Distribution")
plt.show()

# Text Length Distribution  
plt.figure(figsize=(6,4))
sns.histplot(df["text_length"], bins=40, kde=True)
plt.title("Text Length Distribution")
plt.xlabel("Text Length")
plt.show()

# Word Count Distribution 
plt.figure(figsize=(6,4))
sns.histplot(df["word_count"], bins=40, kde=True)
plt.title("Word Count Distribution")
plt.xlabel("Word Count")
plt.show()

In [ ]:
## Bivariate

# Status vs Text Length
plt.figure(figsize=(6,4))
sns.boxplot(x="status", y="text_length", data=df)
plt.title("Status vs Text Length")
plt.show()

# Status vs Word Count
plt.figure(figsize=(6,4))
sns.boxplot(x="status", y="word_count", data=df)
plt.title("Status vs Word Count")
plt.show()

# Text Length vs Word Count
plt.figure(figsize=(6,4))
sns.scatterplot(x="text_length", y="word_count", hue="status", data=df)
plt.title("Text Length vs Word Count by Status")
plt.show()

In [ ]:
## Multivariate

plt.figure(figsize=(6,5))
corr_matrix = df[["text_length", "word_count"]].corr()

sns.heatmap(corr_matrix, annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
## TF-IDF

X = df["statement"]
y = df["status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
## Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
## ROC Curve

# Convert labels to binary format
classes = y.unique()
y_test_bin = label_binarize(y_test, classes=classes)
y_score = model.predict_proba(X_test_vec)

plt.figure(figsize=(8,6))

for i in range(len(classes)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, label=f"{classes[i]} (AUC = {roc_auc:.2f})")

plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Multi-Class)")
plt.legend()
plt.show()

In [ ]:
## MYSQL DATABASE CONNECTION

In [ ]:
# =====================================================
# MySQL Database Connection
# =====================================================

import mysql.connector

db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_password",   
    database="mental_health"
)

if db.is_connected():
    print("✅ Connected to MySQL successfully")

db.close()

In [ ]:
# =====================================================
# Insert Data into MySQL
# =====================================================

import csv

db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_password",
    database="mental_health"
)

cursor = db.cursor()

sql = """
INSERT INTO mental_h (statement, status)
VALUES (%s, %s)
"""

with open("data/cleaned_mh.csv", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    next(reader)  # skip header

    for row in reader:
        cursor.execute(sql, row)

db.commit()
cursor.close()
db.close()

print("✅ Data inserted successfully into MySQL.")

In [ ]:
## Key Business Insights

- Depressive posts generally have higher word counts, indicating deeper emotional expression and detailed narration.
- Text length and word count show strong positive correlation, confirming that longer posts contain more expressive content.
- Anxiety-related posts display higher variability in length, suggesting fluctuating emotional intensity.
- Logistic Regression with TF-IDF effectively classifies mental health categories based on textual patterns.
- The model enables early detection of mental health risks from social media text, supporting proactive intervention.